In [1]:
import pandas as pd 

# Search Evaluation

We use the ground truth to evaluate our search engine because, again, if our search only returns the most relevant results, then our RAG will provide better output. 

In [2]:
df_ground_truth = pd.read_csv("data/ground_truth-new.csv")

df_ground_truth.head()

,question,document
0,Can I still join the course if I found it late?,74eb249bbf
1,Is it too late to start this course now?,74eb249bbf
2,"If I join after the course started, can I stil...",74eb249bbf
3,What do I need to do to be eligible for the ce...,74eb249bbf
4,Do I have to submit the project before submiss...,74eb249bbf


In [3]:
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[:5]

[{'question': 'Can I still join the course if I found it late?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start this course now?',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course started, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to be eligible for the certificate if I join late?',
  'document': '74eb249bbf'},
 {'question': 'Do I have to submit the project before submissions close to get certified?',
  'document': '74eb249bbf'}]

Now we download the documents (aka, our knowledge base) and index them.

In [4]:
from utils.ingest import load_faq_data, build_index

raw_documents = load_faq_data()

documents_llm = [document for document in raw_documents if document["course"] == "llm-zoomcamp"]

index = build_index(documents_llm)

Now we create the text search; vector search will come later. 

In [13]:
def text_search(query:str):
    boost_dict = {"question":3.0, "section":0.5}

    results = index.search(query, num_results=5, boost_dict=boost_dict)

    return results

We'll use the question below for evaluation.

In [17]:
q = ground_truth[0]
q

{'question': 'Can I still join the course if I found it late?',
 'document': '74eb249bbf'}

Let's run a search for this question

In [19]:
doc_id = q['document']
results = text_search(query=q['question'])

We'll compare the retrieved document IDs with the correct document ID: 

In [20]:
for result in results: 
    print(f'{result["id"]}=={doc_id}: {result["id"]==doc_id}')

74eb249bbf==74eb249bbf: True
a9353fadfe==74eb249bbf: False
9f689c185f==74eb249bbf: False
977bf7786c==74eb249bbf: False
69d122f12e==74eb249bbf: False


So, 1 in 5 of our results were relevant. We can save these results as a list for future evaluation. 

In [21]:
relevance = [int(result["id"] == doc_id) for result in results]

In [27]:
relevance

[1, 0, 0, 0, 0]

Wrap it all in a function to make it reusable. 

In [28]:
def compute_relevance_text(q):
    doc_id = q['document']
    results = text_search(query=q['question'])

    relevance = [int(result["id"] == doc_id) for result in results]

    return relevance


In [29]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where can I find the livestream link for office hours or workshop sessions before they start?


[1, 0, 0, 0, 0]